# AutoGen Learning Notebook (Local Ollama + `gemma4:e2b`)

This notebook teaches AutoGen section-by-section using your local model.

Learning path:
1. Environment checks
2. Create a reusable model client
3. Build your first AutoGen agent
4. Improve behavior with system prompts
5. Run a multi-agent team
6. Mini project challenge

## 0) Install (Run Once)
If this kernel does not have AutoGen installed, run this cell.

In [ ]:
# %pip install -U autogen-agentchat "autogen-ext[ollama]"

## 1) Environment Check
We verify Python, packages, Ollama service, and model availability.

In [3]:
import importlib
import platform
import subprocess

print(f"Python: {platform.python_version()}")
for pkg in ["autogen_agentchat", "autogen_ext"]:
    m = importlib.import_module(pkg)
    print(f"{pkg}: {getattr(m, '__version__', 'version not exposed')}")

model_tag = "gemma4:e2b"
models = subprocess.check_output(["ollama", "list"], text=True)
print("\nInstalled Ollama models:")
print(models)
if model_tag not in models:
    raise RuntimeError(f"Model {model_tag} not found. Run: ollama pull {model_tag}")

print(f"Model check passed: {model_tag}")

Python: 3.13.3
autogen_agentchat: 0.7.5
autogen_ext: 0.7.5

Installed Ollama models:
NAME                       ID              SIZE      MODIFIED      
gemma4:e2b                 7fbdbf8f5e45    7.2 GB    20 hours ago     
nemotron-3-nano:4b         6cc467f05439    2.8 GB    13 days ago      
smollm2:latest             cef4a1e09247    1.8 GB    3 months ago     
gemma3:1b                  8648f39daa8f    815 MB    8 months ago     
nomic-embed-text:latest    0a109f422b47    274 MB    12 months ago    
llama3.2:latest            a80c4f17acd5    2.0 GB    14 months ago    

Model check passed: gemma4:e2b


## 2) Create a Reusable AutoGen Ollama Client
`gemma4:e2b` is not yet in AutoGen's built-in Ollama model map, so we pass `model_info` explicitly.

In [4]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

MODEL_TAG = "gemma4:e2b"
MODEL_INFO = {
    "vision": False,
    "function_calling": False,
    "json_output": False,
    "family": "unknown",
    "structured_output": False,
}

def build_client(model: str = MODEL_TAG) -> OllamaChatCompletionClient:
    return OllamaChatCompletionClient(
        model=model,
        model_info=MODEL_INFO,
        host="http://localhost:11434",
    )

print("Client helper ready.")

Client helper ready.


## 3) Your First AutoGen Agent
Create an `AssistantAgent`, run one task, inspect output.

In [5]:
from autogen_agentchat.agents import AssistantAgent

client = build_client()
agent = AssistantAgent(
    name="local_assistant",
    model_client=client,
    system_message="You are concise and practical.",
)

result = await agent.run(task="Explain AutoGen in 3 bullet points for a beginner.")
print(result.messages[-1].content)

await client.close()

*   **Multi-Agent System:** AutoGen allows you to create and orchestrate multiple, specialized AI agents that can communicate with each other to solve complex problems.
*   **Role-Based Collaboration:** You assign specific roles (e.g., Coder, Tester, Critic) to these agents, enabling them to collaborate step-by-step to complete a larger task.
*   **Automated Workflow:** Instead of writing one long prompt, you define how the agents interact, letting the system manage the conversation and iteration needed to reach the final goal.


## 4) Prompt Steering With System Messages
Same task, different system prompts, different behavior.

In [6]:
async def run_with_style(style: str, task: str):
    client = build_client()
    agent = AssistantAgent(
        name="styled_agent",
        model_client=client,
        system_message=style,
    )
    result = await agent.run(task=task)
    await client.close()
    return result.messages[-1].content

task = "Teach AutoGen setup steps in plain language."
style_1 = "You are a teacher. Use numbered steps."
style_2 = "You are a strict reviewer. List common mistakes and fixes."

out1 = await run_with_style(style_1, task)
out2 = await run_with_style(style_2, task)

print("=== Teacher Style ===\n")
print(out1)
print("\n=== Reviewer Style ===\n")
print(out2)

=== Teacher Style ===

Hello! I'm happy to teach you about setting up AutoGen. Think of setting up AutoGen like preparing a workshop before you start building. We need to make sure all our tools are ready before we can start the actual building process.

Here are the step-by-step instructions to get your AutoGen environment ready, explained in plain language.

---

### Phase 1: Preparing Your Computer (The Foundation)

**Step 1: Make Sure You Have Python Installed**
Before we do anything else, you need the main tool. Python is the language that AutoGen is built on.
*   **What to do:** Go to the official Python website and download the latest version. Make sure you select the option to add Python to your system's PATH during installation (this is usually an easy checkbox).

**Step 2: Create an Isolated Workspace (Virtual Environment)**
It is a very good practice to keep the libraries for different projects separate. This prevents conflicts.
*   **What to do:** Open your computer's Termi

## 5) Multi-Agent Team (Round Robin)
Here two agents collaborate: one drafts, one critiques.

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat

client = build_client()

planner = AssistantAgent(
    name="planner",
    model_client=client,
    system_message="Create a compact plan with max 4 bullets.",
)
reviewer = AssistantAgent(
    name="reviewer",
    model_client=client,
    system_message="Critique the plan and improve clarity.",
)

team = RoundRobinGroupChat([planner, reviewer], max_turns=4)
team_result = await team.run(task="Build a 7-day roadmap to learn AutoGen basics.")

for i, msg in enumerate(team_result.messages):
    print(f"\n[{i}] {msg.source}:")
    print(msg.content)

await client.close()

## 6) Mini Project: Build a Local Study Coach
Try changing the task and prompts below to create your own agent behavior.

In [ ]:
client = build_client()
coach = AssistantAgent(
    name="study_coach",
    model_client=client,
    system_message=(
        "You are a study coach. Give practical, short plans with milestones and checks."
    ),
)

user_goal = "I can spend 45 minutes/day. Help me learn AutoGen in 2 weeks."
coach_result = await coach.run(task=user_goal)
print(coach_result.messages[-1].content)

await client.close()

## 7) What To Learn Next
- Add tools once you switch to a model with stronger tool-calling support.
- Try `run_stream(...)` for token streaming UX.
- Build a domain agent (coding assistant, notes summarizer, support bot).